# Keystone-Cowork — Single-GPU Engine (Blackwell RTX PRO 6000 / A100)

**Hardware Target:** 1x NVIDIA RTX PRO 6000 Blackwell Server Edition (96 GB VRAM) / NVIDIA A100-SXM4-80GB
- **Cognition Engine:** `llama.cpp` (`llama-cli` + `llama-server`, Native CUDA + Flash Attention, `-ngl 999`, Q8 KV Cache, 16 slots) running `OBLITERATUS/Qwen3.8-27B-OBLITERATED` (`Qwen3.8-27B-OBLITERATED-Q8_0.gguf`)
- **Creative Video & Audio Engine:** `ComfyUI` + `ComfyUI-KJNodes` + `ComfyUI-VideoHelperSuite` + `ComfyUI-Manager` + `SageAttention` running `SexGod1979/PinkCherry-NSFW_MiniMax-H3` (`PinkCherry_hybrid_v1.8_minimax_h3_pruned_bf16.safetensors` & `int8`) + MiniMax-H3 VAEs, TAEH3, 3D-Conv Latent Upscaler, Qwen3-VL 32B Text Encoders, and Turbo LoRAs.

## 1. Compile & Install llama-cli and llama-server (Native CUDA + Flash Attention)

In [ ]:
!nvidia-smi
import os
if not os.path.exists("/content/llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
%cd /content/llama.cpp

# Clean previous build cache
!rm -rf build

# Target Blackwell RTX PRO 6000 (sm_120) and A100 (sm_80) with Q8/16 Flash Attention
!cmake -B build \
  -DGGML_CUDA=ON \
  -DCMAKE_CUDA_ARCHITECTURES="80;120" \
  -DGGML_CUDA_FA_QUANTS=all \
  -DGGML_CCACHE=OFF \
  -DBUILD_SHARED_LIBS=OFF \
  -DCMAKE_BUILD_TYPE=Release

!cmake --build build --config Release -j$(nproc) --target llama-cli llama-server llama-bench llama-quantize

# Stop any running server to release the file lock before copying
!pkill -f llama-server || true
import time
time.sleep(2)

# Install binaries globally into /usr/local/bin
!cp build/bin/llama-cli build/bin/llama-server build/bin/llama-bench build/bin/llama-quantize /usr/local/bin/
!llama-cli --list-devices

## 2. Authenticate HF_TOKEN & Pull OBLITERATUS/Qwen3.8-27B-OBLITERATED (Text/Reasoning Model)

In [ ]:
!pip install -q hf_transfer
import os
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["GGML_CUDA_P2P"] = "1"

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN", "")

os.environ["HF_TOKEN"] = hf_token

REPO_ID = "OBLITERATUS/Qwen3.8-27B-OBLITERATED"
target_file = "Qwen3.8-27B-OBLITERATED-Q8_0.gguf"

os.makedirs("/content/models", exist_ok=True)

print(f"Downloading {target_file} from {REPO_ID}...")
MODEL_PATH = hf_hub_download(
    repo_id=REPO_ID,
    filename=target_file,
    local_dir="/content/models",
    token=hf_token or None
)

print(f"\nMODEL READY AT: {MODEL_PATH}")

## 3. Engine Configuration (Blackwell RTX PRO 6000 / A100 GPU)

In [ ]:
# Single Blackwell RTX PRO 6000 / A100 Server Edition Engine Configuration
LLAMA_CONFIG = {
    # Single GPU Distribution
    "-ngl": "999",               # Offload 100% of layers to the GPU (96 GB VRAM)
    "--flash-attn": "on",        # Enable Flash Attention kernels

    # Context & KV Cache (Q8 and 16 only)
    "-c": "32768",               # 32k context
    "-ctk": "q8_0",              # Q8 K-cache
    "-ctv": "q8_0",              # Q8 V-cache

    # Batching & Concurrency for Blackwell RTX PRO 6000 (sm_120)
    "-b": "2048",
    "-ub": "2048",
    "-np": "16",                 # 16 concurrent request slots
    "--cont-batching": "",

    # Tool Calling & Sampling Overrides
    "--jinja": "",
    "--temp": "0.2",
    "--min-p": "0.05",
    "--repeat-penalty": "1.05"
}

def build_flags(config_dict):
    flags = []
    for k, v in config_dict.items():
        flags.append(k.strip())
        if v != "":
            flags.append(str(v))
    return flags

print("Active CLI Flags:\n", " ".join(build_flags(LLAMA_CONFIG)))

## 4. Single-Shot Hardware & Model Verification via llama-cli

In [ ]:
import shlex

# Configure Q8 and 16 only
LLAMA_CONFIG["-ctk"] = "q8_0"
LLAMA_CONFIG["-ctv"] = "q8_0"

test_config = LLAMA_CONFIG.copy()
test_config.pop("--cont-batching", None)
test_config.pop("-np", None)

print(f"Active Cache Quantization: -ctk {LLAMA_CONFIG['-ctk']} | -ctv {LLAMA_CONFIG['-ctv']}")

cli_cmd = [
    "llama-cli",
    "-m", MODEL_PATH,
    "--single-turn",
    "-p", "Keystone-Cowork RTX PRO 6000 check with Q8 cache: reply READY in one sentence.",
    "-n", "128"
] + build_flags(test_config)

!{shlex.join(cli_cmd)}

## 5. Launch Live OpenAI Endpoint (/v1) & Print Keystone-Cowork Config

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

import subprocess
import time
import re
import urllib.request

MODEL_PATH = "/content/models/Qwen3.8-27B-OBLITERATED-Q8_0.gguf"
model_alias = "OBLITERATUS/Qwen3.8-27B-OBLITERATED:fastest"

# Lock to Q8 and 16 only
LLAMA_CONFIG["-ctk"] = "q8_0"
LLAMA_CONFIG["-ctv"] = "q8_0"
LLAMA_CONFIG.pop("--no-mmap", None)
LLAMA_CONFIG.pop("--mlock", None)
LLAMA_CONFIG.pop("--load-mode", None)

# Stop any previously running llama-server or tunnel on port 8081
subprocess.run(["pkill", "-f", "llama-server.*8081"], check=False)
subprocess.run(["pkill", "-f", "cloudflared.*8081"], check=False)
time.sleep(2)

# Start llama-server on port 8081
server_cmd = [
    "llama-server",
    "-m", MODEL_PATH,
    "--alias", model_alias,
    "--host", "0.0.0.0",
    "--port", "8081",
    "--api-key", "keystone-live-key",
    "--metrics",
    "--slots"
] + build_flags(LLAMA_CONFIG)

server_log = open("/content/llama_server.log", "w")
server_proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT)

# Launch Cloudflare Tunnel for port 8081
tunnel_log_path = "/content/cloudflared_8081.log"
tunnel_log = open(tunnel_log_path, "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8081"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

public_url = None
for _ in range(30):
    time.sleep(1)
    with open(tunnel_log_path, "r") as f:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
        if match:
            public_url = match.group(0)
            break

# Wait for llama-server health endpoint
print(f"RTX PRO 6000 Config Active | GPU Layers: {LLAMA_CONFIG.get('-ngl', '999')} | Slots: {LLAMA_CONFIG.get('-np', '16')} | KV Cache: Q8")
print("Waiting for llama-server to load model on port 8081...")
for i in range(120):
    if server_proc.poll() is not None:
        print("llama-server exited early! Log output:")
        print(open("/content/llama_server.log").read())
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8081/health", timeout=2) as resp:
            if resp.status == 200:
                print("llama-server is LIVE and healthy!")
                break
    except Exception:
        time.sleep(2)

print("=" * 72)
print(f"KEYSTONE-COWORK LIVE OPENAI ENDPOINT: {public_url}/v1")
print(f"MODEL ID:                             {model_alias}")
print(f"API KEY:                              keystone-live-key")
print("=" * 72)
print("\nPaste into keystone-cowork .env:")
print(f"OPENAI_BASE_URL={public_url}/v1")
print(f"OPENAI_API_KEY=keystone-live-key")
print(f"OPENAI_MODEL={model_alias}")
print("=" * 72)

## Server Benchmarking: Concurrent Requests Test

In [ ]:
import asyncio
import aiohttp
import time

API_URL = "http://127.0.0.1:8081/v1/chat/completions"
API_KEY = "keystone-live-key"
CONCURRENT_REQUESTS = 4
PROMPT = "Explain the importance of optimization in llama.cpp in two sentences."

async def send_request(session, request_id):
    payload = {
        "model": "OBLITERATUS/Qwen3.8-27B-OBLITERATED:fastest",
        "messages": [{"role": "user", "content": PROMPT}],
        "temperature": 0.2,
        "max_tokens": 128,
        "chat_template_kwargs": {"enable_thinking": False}
    }
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    start_time = time.time()
    async with session.post(API_URL, json=payload, headers=headers) as response:
        if response.status == 200:
            result = await response.json()
            duration = time.time() - start_time
            msg = result["choices"][0]["message"]
            text = (msg.get("content") or msg.get("reasoning_content") or "").strip()
            tokens = result.get("usage", {}).get("completion_tokens", 0)
            tps = tokens / duration if duration > 0 else 0
            print(f"[Request {request_id}] Success | Time: {duration:.2f}s | Tokens: {tokens} | Speed: {tps:.2f} t/s\n-> Response: {text}\n")

async def main():
    t0 = time.time()
    async with aiohttp.ClientSession() as session:
        await asyncio.gather(*(send_request(session, i) for i in range(CONCURRENT_REQUESTS)))
    print(f"Benchmark completed in {time.time() - t0:.2f} seconds.")

await main()

## 6. Install ComfyUI, Custom Nodes (KJNodes & VideoHelperSuite) & Download All MiniMax H3 / PinkCherry Safetensors

In [ ]:
import os
import json
import shutil
import subprocess
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
hf_token = os.environ.get("HF_TOKEN")

# 1. Clone ComfyUI & Required Custom Nodes (KJNodes, VideoHelperSuite, ComfyUI-Manager)
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

custom_nodes = {
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "ComfyUI-VideoHelperSuite": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "ComfyUI-Manager": "https://github.com/ltdrdata/ComfyUI-Manager.git"
}
for name, url in custom_nodes.items():
    target = f"/content/ComfyUI/custom_nodes/{name}"
    if not os.path.exists(target):
        !git clone --depth 1 {url} {target}
    req_file = os.path.join(target, "requirements.txt")
    if os.path.exists(req_file):
        !pip install -q -r {req_file}

!pip install -q sageattention

base_models = "/content/ComfyUI/models"
for sub in ["diffusion_models", "checkpoints", "vae", "vae_approx", "text_encoders", "loras", "embeddings", "latent_upscale_models"]:
    os.makedirs(os.path.join(base_models, sub), exist_ok=True)

def download_flat(repo_id, filename, target_dir):
    """Downloads a file from Hugging Face Hub and places it directly in target_dir."""
    basename = os.path.basename(filename)
    dest_path = os.path.join(target_dir, basename)
    if os.path.exists(dest_path):
        print(f"✓ Already present: {dest_path}")
        return dest_path
    print(f"Downloading {filename} from {repo_id} -> {dest_path}...")
    downloaded = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir="/content/ComfyUI/hf_cache",
        token=hf_token
    )
    shutil.move(downloaded, dest_path)
    return dest_path

def ensure_symlink(target_path, link_path):
    os.makedirs(os.path.dirname(link_path), exist_ok=True)
    if not os.path.exists(link_path):
        if os.path.islink(link_path):
            os.unlink(link_path)
        os.symlink(target_path, link_path)

# 2. PinkCherry MiniMax-H3 Diffusion Models (1.8 Hybrid bf16 + int8)
diff_dir = os.path.join(base_models, "diffusion_models")
pinkcherry_bf16 = download_flat(
    "SexGod1979/PinkCherry-NSFW_MiniMax-H3",
    "1.8-hybrid/PinkCherry_hybrid_v1.8_minimax_h3_pruned_bf16.safetensors",
    diff_dir
)
pinkcherry_int8 = download_flat(
    "SexGod1979/PinkCherry-NSFW_MiniMax-H3",
    "1.8-hybrid/PinkCherry_hybrid_v1.8_int8_minimax_h3_pruned.safetensors",
    diff_dir
)

# Create aliases expected by the workflows and CheckpointLoader
ensure_symlink(pinkcherry_bf16, os.path.join(base_models, "checkpoints", os.path.basename(pinkcherry_bf16)))
ensure_symlink(pinkcherry_int8, os.path.join(diff_dir, "PinkCherry_hybrid_v1.8_int8_minimax_h3.safetensors"))
ensure_symlink(pinkcherry_bf16, os.path.join(diff_dir, "minimax_h3_fl2va_int8_convrot.safetensors"))
ensure_symlink(pinkcherry_bf16, os.path.join(diff_dir, "minimax_h3_fl2va_pruned_int8_convrot.safetensors"))
ensure_symlink(pinkcherry_bf16, os.path.join(diff_dir, "minimax_h3_ref2va_pruned_int8_convrot.safetensors"))

# 3. MiniMax-H3 Video & Audio VAEs + TAEH3 Preview Decoder + 3D-Conv Latent Upscaler
download_flat("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_fp16.safetensors", os.path.join(base_models, "vae"))
download_flat("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", os.path.join(base_models, "vae"))
download_flat("t8star/Taeh3-Comfy", "models/vae_approx/taeh3.safetensors", os.path.join(base_models, "vae_approx"))
download_flat("t8star/Taeh3-Comfy", "models/vae_approx/taeh3_2d_kijai.safetensors", os.path.join(base_models, "vae_approx"))
download_flat("LBH-123-AI/Minimax_h3_latent_Upscaler", "minimax_h3_latent_upscaler_3d_conv_v1_bf16.safetensors", os.path.join(base_models, "latent_upscale_models"))

# 4. MiniMax-H3 Qwen3-VL 32B Text Encoders
download_flat("Comfy-Org/MiniMax-H3", "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors", os.path.join(base_models, "text_encoders"))
download_flat("Comfy-Org/MiniMax-H3", "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", os.path.join(base_models, "text_encoders"))

# 5. MiniMax-H3 Turbo LoRAs & Embeddings
lora_dir = os.path.join(base_models, "loras")
lora_8step = download_flat("Comfy-Org/MiniMax-H3", "loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", lora_dir)
lora_4step = download_flat("Comfy-Org/MiniMax-H3", "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors", lora_dir)
ensure_symlink(lora_8step, os.path.join(lora_dir, "MiniMaxh3 Turbo", os.path.basename(lora_8step)))
download_flat("Comfy-Org/MiniMax-H3", "embeddings/minimaxh3_art_is_explosion.safetensors", os.path.join(base_models, "embeddings"))

# 6. Populate sample reference inputs in /content/ComfyUI/input so LoadImage/LoadAudio/VHS_LoadVideo validate cleanly
input_dir = "/content/ComfyUI/input"
os.makedirs(input_dir, exist_ok=True)
sample_mp4 = download_flat("SexGod1979/PinkCherry-NSFW_MiniMax-H3", "1.8-hybrid/Test__00347-audio.mp4", input_dir)
for vid_alias in ["3e785606-83ea-4a5f-8dde-437d60f41b71.mp4", "ref_clip.mp4", "source.mp4", "tail_22frames.mp4"]:
    ensure_symlink(sample_mp4, os.path.join(input_dir, vid_alias))
for img_name in ["49697335_091_3b3d.jpg", "Krea2__00319_.png", "0027.jpg", "ref_1.png", "ref_2.png"]:
    img_dest = os.path.join(input_dir, img_name)
    if not os.path.exists(img_dest):
        subprocess.run(["ffmpeg", "-y", "-i", sample_mp4, "-frames:v", "1", img_dest], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 7. Download & Patch Workflows for Linux Paths, Installed Models, and Complete Node Links
workflow_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(workflow_dir, exist_ok=True)
for wf in ["1.8-hybrid/PinkCherry_ref2va_turbo_lightx2v_workflow.json", "v1.5-fl2va/MiniMaxH3_PinkCherryV15_Fla2va_wTurbo.json"]:
    wf_path = download_flat("SexGod1979/PinkCherry-NSFW_MiniMax-H3", wf, workflow_dir)
    with open(wf_path, "r") as f:
        wf_text = f.read()
    wf_text = wf_text.replace("MiniMaxh3 Turbo\\minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors")
    wf_text = wf_text.replace("PinkCherry_hybrid_v1.8_int8_minimax_h3.safetensors", "PinkCherry_hybrid_v1.8_minimax_h3_pruned_bf16.safetensors")
    wf_text = wf_text.replace("minimax_h3_fl2va_int8_convrot.safetensors", "PinkCherry_hybrid_v1.8_minimax_h3_pruned_bf16.safetensors")
    wf_text = wf_text.replace("minimax_h3_fl2va_pruned_int8_convrot.safetensors", "PinkCherry_hybrid_v1.8_minimax_h3_pruned_bf16.safetensors")
    wf_text = wf_text.replace("REf2va\\TEST_", "REf2va/TEST_")
    wf_text = wf_text.replace("MiniMaxH3_BaseLora\\Test_", "MiniMaxH3_BaseLora/Test_")
    wf_data = json.loads(wf_text)
    if "MiniMaxH3_PinkCherryV15" in wf_path and not any(l[0] == 391 for l in wf_data.get("links", [])):
        wf_data["links"].append([391, 168, 0, 119, 0, "IMAGE"])
        for n in wf_data.get("nodes", []):
            if n["id"] == 168 and 391 not in n["outputs"][0]["links"]:
                n["outputs"][0]["links"].append(391)
            elif n["id"] == 119:
                n["inputs"][0]["link"] = 391
        wf_data["last_link_id"] = max(wf_data.get("last_link_id", 390), 391)
    with open(wf_path, "w") as f:
        json.dump(wf_data, f, indent=2)

!rm -rf /content/ComfyUI/hf_cache
print("\n✅ ComfyUI custom nodes, MiniMax-H3/PinkCherry safetensors, 3D-Conv Latent Upscaler, and patched workflows are ready!")

## 7. Launch ComfyUI & Tunnel (Port 8188)

In [ ]:
import subprocess
import time
import re
import urllib.request

# Stop any previously running ComfyUI or tunnel on port 8188
subprocess.run(["pkill", "-f", "main.py.*8188"], check=False)
subprocess.run(["pkill", "-f", "cloudflared.*8188"], check=False)
time.sleep(1)

# Launch ComfyUI in the background on port 8188
print("Starting ComfyUI...")
comfy_log = open("/content/comfyui.log", "w")
comfy_proc = subprocess.Popen(
    ["python3", "main.py", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=comfy_log,
    stderr=subprocess.STDOUT
)

# Wait for ComfyUI to listen locally on port 8188
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188", timeout=2) as resp:
            if resp.status == 200:
                print("ComfyUI is listening locally on port 8188!")
                break
    except Exception:
        time.sleep(1)

# Launch a second Cloudflare Tunnel for ComfyUI
print("Starting Cloudflare Tunnel for ComfyUI...")
tunnel_comfy_log_path = "/content/cloudflared_8188.log"
tunnel_comfy_log = open(tunnel_comfy_log_path, "w")
tunnel_proc_comfy = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
    stdout=tunnel_comfy_log,
    stderr=subprocess.STDOUT
)

public_url_comfy = None
for _ in range(30):
    time.sleep(1)
    with open(tunnel_comfy_log_path, "r") as f:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
        if match:
            public_url_comfy = match.group(0)
            break

print("=" * 72)
print(f"COMFYUI LIVE ENDPOINT: {public_url_comfy}")
print("=" * 72)

## 8. Connect to Keystone Repository (.env Setup)

In [ ]:
import os

repo_url = "https://github.com/jordandavid1/Keystone.git"
repo_dir = "/content/Keystone"

if not os.path.exists(repo_dir):
    !git clone {repo_url} {repo_dir}

env_path = os.path.join(repo_dir, ".env")
env_content = f"""# Keystone-Cowork Runtime Configuration
OPENAI_BASE_URL={public_url}/v1
OPENAI_API_KEY=keystone-live-key
OPENAI_MODEL={model_alias}

# MiniMax H3 / PinkCherry 1.8 ComfyUI Runtime
KEYSTONE_COMFY_URL=http://127.0.0.1:8188
KEYSTONE_COMFY_PUBLIC_URL={public_url_comfy}
KEYSTONE_COMFY_WORKFLOW=workflows/h3-ref2va.json
KEYSTONE_H3_CHECKPOINT=PinkCherry_hybrid_v1.8_minimax_h3_pruned_bf16.safetensors
KEYSTONE_H3_REPO=SexGod1979/PinkCherry-NSFW_MiniMax-H3
KEYSTONE_COMFY_DURATION=5
KEYSTONE_COMFY_STEPS=8
KEYSTONE_COMFY_MEGAPIXELS=0.5
KEYSTONE_COMFY_ASPECT=16:9
KEYSTONE_COMFY_TIMEOUT=1800
"""

with open(env_path, "w") as f:
    f.write(env_content)

print(f"Successfully updated {env_path} with OpenAI & ComfyUI H3 endpoints!")

## 9. Verify ComfyUI Connectivity

In [ ]:
import requests
import json
import os
import time

print(f"Testing ComfyUI tunnel URL: {public_url_comfy}")
for attempt in range(10):
    try:
        response = requests.get(public_url_comfy, timeout=10)
        if response.status_code == 200:
            print("✅ ComfyUI is reachable via the public Cloudflare tunnel!")
            break
    except Exception:
        pass
    time.sleep(2)

# Audit all workflow nodes, links, required inputs, and model dropdowns against live ComfyUI /object_info
obj_info = requests.get("http://127.0.0.1:8188/object_info", timeout=10).json()
wf_dir = "/content/ComfyUI/user/default/workflows"
for wf_file in sorted(os.listdir(wf_dir)):
    if not wf_file.endswith(".json"):
        continue
    with open(os.path.join(wf_dir, wf_file), "r") as f:
        data = json.load(f)
    all_nodes = data.get("nodes", [])
    missing_nodes = [n["type"] for n in all_nodes if n.get("type") not in obj_info and not n.get("type", "").startswith(("Note", "MarkdownNote", "Reroute", "PrimitiveNode"))]
    print(f"=== UI Workflow Audit:  {wf_file:48s} | Nodes: {len(all_nodes):2d} | Missing: {missing_nodes} ===")

## 🔍 Diagnostic: Check Live Process Logs

In [ ]:
import os

def print_log(title, path):
    print("=" * 40)
    print(f" Logs for: {title} ({path}) ")
    print("=" * 40)
    if os.path.exists(path):
        with open(path, "r") as f:
            lines = f.readlines()
            for line in lines[-30:]:
                print(line.strip())
    else:
        print("Log file not found yet.")
    print("\n")

print_log("llama-server", "/content/llama_server.log")
print_log("ComfyUI", "/content/comfyui.log")

## 10. Atomic Chat Bridge (Gradio UI)

In [ ]:
!pip install -q gradio openai
import gradio as gr
from openai import OpenAI

client_local = OpenAI(
    base_url="http://127.0.0.1:8081/v1",
    api_key="keystone-live-key"
)

def atomic_chat(message, history):
    messages = []
    for turn in history:
        if isinstance(turn, dict):
            messages.append({"role": turn.get("role", "user"), "content": turn.get("content", "")})
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            messages.append({"role": "user", "content": turn[0]})
            if turn[1]:
                messages.append({"role": "assistant", "content": turn[1]})
    messages.append({"role": "user", "content": message})
    resp = client_local.chat.completions.create(
        model="OBLITERATUS/Qwen3.8-27B-OBLITERATED:fastest",
        messages=messages,
        temperature=0.2,
        max_tokens=1024
    )
    return resp.choices[0].message.content

print("Launching Atomic Chat Bridge...")
gr.ChatInterface(fn=atomic_chat, title="KEYSTONE Atomic Chat Bridge (OBLITERATUS/Qwen3.8-27B-OBLITERATED)").launch(share=True)

## 14. Launch KEYSTONE Multi-Mode Creative Studio (T2V / Ref2Video / V2V / Extend / 2x Latent Upscale / 30s–60s Long-Form)

In [ ]:
import sys
import subprocess
import time
import re
import requests

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "greenlet", "aiosqlite", "python-multipart"], check=False)

# Kill any previous Keystone API or tunnel on port 8000
subprocess.run(["pkill", "-f", "uvicorn apps.api.src"], check=False)
subprocess.run(["pkill", "-f", "cloudflared.*8000"], check=False)
time.sleep(1)

# 1. Start KEYSTONE FastAPI Server on port 8000
keystone_log = open("/content/keystone_api.log", "w")
keystone_proc = subprocess.Popen(
    ["python3", "-m", "uvicorn", "apps.api.src.server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/Keystone",
    stdout=keystone_log,
    stderr=subprocess.STDOUT,
)

for _ in range(40):
    try:
        r = requests.get("http://127.0.0.1:8000/v1/health", timeout=2)
        if r.status_code == 200:
            break
    except Exception:
        time.sleep(0.5)

# 2. Launch Cloudflare Tunnel for KEYSTONE Studio
tunnel_log = "/content/keystone_tunnel.log"
with open(tunnel_log, "w") as f:
    subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=f,
        stderr=subprocess.STDOUT,
    )

public_url_keystone = None
for _ in range(30):
    time.sleep(1)
    with open(tunnel_log, "r") as f:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
        if match:
            public_url_keystone = match.group(0)
            break

health = requests.get("http://127.0.0.1:8000/v1/creative/h3", timeout=5).json()
workflows = requests.get("http://127.0.0.1:8000/v1/creative/h3/workflows", timeout=5).json()

print("=" * 78)
print(f"🚀 KEYSTONE Creative Workspace Live: {public_url_keystone}/workspace")
print(f"🔗 KEYSTONE Public API Health:       {public_url_keystone}/v1/health")
print(f"🎬 Supported Creative Modes:         {health['runtime']['modes']}")
print(f"📦 Registered H3 Workflows:          {list(workflows['workflows'].keys())}")
print(f"🟢 ComfyUI Backend Reachable:        {health['health']['reachable']} (HTTP {health['health'].get('status')})")
print("=" * 78)